# Final Insights\n
\n
Project: Albania Brain Drain Analysis\n

# Final Insights and Reporting

## Description

This notebook generates the final written outputs for the **Albania Brain Drain Analysis** project. It uses the processed datasets and previously calculated results to create the final executive summary and research report.

The notebook reads cleaned data from the `data/processed/` folder, extracts key values such as population change, net migration, unemployment, GDP growth, residence permits, demographic indicators, correlation results, and forecast outputs, then writes the final reports into the `reports/` folder.

## Goal

The goal of this notebook is to transform the completed analysis into clear final project deliverables.

By the end of this notebook, the project will have:

* A completed `executive_summary.md`
* A completed `research_report.md`
* Calculated findings included in the written report
* Clear explanation of results, limitations, and conclusions
* Final documentation ready for GitHub, academic submission, and portfolio presentation

This notebook represents the final reporting step of the data analytics workflow.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Detect project root
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
REPORTS_DIR = ROOT_DIR / "reports"
POWERBI_DIR = ROOT_DIR / "dashboard" / "powerbi_data"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_if_exists(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

def fmt_number(value):
    if pd.isna(value):
        return "not available"
    return f"{value:,.0f}"

def fmt_decimal(value, digits=2):
    if pd.isna(value):
        return "not available"
    return f"{value:,.{digits}f}"

def fmt_percent(value, digits=2):
    if pd.isna(value):
        return "not available"
    return f"{value:.{digits}f}%"

def latest_non_null(df, column):
    if column not in df.columns:
        return np.nan, np.nan
    temp = df[["year", column]].dropna()
    if temp.empty:
        return np.nan, np.nan
    row = temp.sort_values("year").iloc[-1]
    return int(row["year"]), row[column]

def value_for_year(df, column, year):
    if column not in df.columns:
        return np.nan
    temp = df[df["year"] == year]
    if temp.empty:
        return np.nan
    return temp[column].iloc[0]

def safe_markdown(df):
    if df.empty:
        return "No data available."
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

# Load main dataset
main_with_demo = read_csv_if_exists(PROCESSED_DIR / "master_analysis_dataset_with_demographics.csv")
main_basic = read_csv_if_exists(PROCESSED_DIR / "master_analysis_dataset.csv")

main = main_with_demo if not main_with_demo.empty else main_basic
main = main.sort_values("year").reset_index(drop=True)

# Load support datasets
migration_destinations = read_csv_if_exists(POWERBI_DIR / "migration_destinations_dashboard.csv")
residence_reason = read_csv_if_exists(POWERBI_DIR / "residence_permits_reason_dashboard.csv")
correlations = read_csv_if_exists(PROCESSED_DIR / "statistical_correlation_results.csv")
forecast_2035 = read_csv_if_exists(PROCESSED_DIR / "forecast_2035_summary.csv")
best_models = read_csv_if_exists(PROCESSED_DIR / "forecast_best_models.csv")

# Core values
pop_2000 = value_for_year(main, "population_total", 2000)
latest_pop_year, latest_pop = latest_non_null(main, "population_total")

pop_change = latest_pop - pop_2000 if not pd.isna(latest_pop) and not pd.isna(pop_2000) else np.nan
pop_change_pct = (pop_change / pop_2000) * 100 if not pd.isna(pop_change) and pop_2000 != 0 else np.nan

latest_mig_year, latest_net_migration = latest_non_null(main, "net_migration")
latest_unemp_year, latest_unemployment = latest_non_null(main, "unemployment_total_percent")
latest_gdp_year, latest_gdp_growth = latest_non_null(main, "gdp_growth_annual_percent")
latest_gdppc_year, latest_gdp_pc = latest_non_null(main, "gdp_per_capita_current_usd")
latest_permits_year, latest_permits = latest_non_null(main, "total_first_residence_permits")

# Worst migration pressure year
if "net_migration" in main.columns:
    mig_temp = main[["year", "net_migration"]].dropna().sort_values("net_migration")
    if not mig_temp.empty:
        worst_mig_year = int(mig_temp.iloc[0]["year"])
        worst_mig_value = mig_temp.iloc[0]["net_migration"]
    else:
        worst_mig_year = "not available"
        worst_mig_value = np.nan
else:
    worst_mig_year = "not available"
    worst_mig_value = np.nan

# Top destination
if not migration_destinations.empty and "total_immigration_recorded" in migration_destinations.columns:
    top_dest_row = migration_destinations.sort_values("total_immigration_recorded", ascending=False).iloc[0]
    top_destination = top_dest_row["reporting_country"]
    top_destination_value = top_dest_row["total_immigration_recorded"]
else:
    top_destination = "not available"
    top_destination_value = np.nan

# Demographic values
latest_working_year, latest_working_age = latest_non_null(main, "working_age_population_15_64")
latest_elderly_year, latest_elderly_share = latest_non_null(main, "elderly_share_65_plus_percent")
latest_dependency_year, latest_dependency = latest_non_null(main, "dependency_ratio_total")

# Correlation table summary
correlation_brief = correlations.copy()
if not correlation_brief.empty:
    keep_cols = [
        col for col in [
            "x_variable",
            "y_variable",
            "n_observations",
            "pearson_r",
            "pearson_p_value",
            "spearman_rho",
            "spearman_p_value",
            "significance_5_percent"
        ]
        if col in correlation_brief.columns
    ]
    correlation_brief = correlation_brief[keep_cols]

# Forecast table
forecast_brief = forecast_2035.copy()
if not forecast_brief.empty:
    keep_cols = [
        col for col in [
            "indicator_label",
            "model",
            "year",
            "forecast_value"
        ]
        if col in forecast_brief.columns
    ]
    forecast_brief = forecast_brief[keep_cols]

# Residence permit latest reason table
if not residence_reason.empty and "year" in residence_reason.columns:
    latest_reason_year = int(residence_reason["year"].max())
    latest_reason_table = (
        residence_reason[residence_reason["year"] == latest_reason_year]
        .sort_values("total_permits", ascending=False)
    )
else:
    latest_reason_year = "not available"
    latest_reason_table = pd.DataFrame()

# Executive Summary
executive_summary = f"""# Executive Summary

## Project Title

**Albania’s Brain Drain: A Data-Driven Analysis of Migration, Employment and Economic Development**

## Purpose of the Project

This project analyzes Albania’s brain drain problem using public data from the World Bank, Eurostat, INSTAT-related reports, and migration datasets. The goal is to understand how population decline, migration flows, residence permits, unemployment, GDP growth, and demographic structure are connected.

The project follows a complete data analytics workflow: raw data collection, cleaning, transformation, PostgreSQL database design, exploratory analysis, statistical analysis, forecasting, and Power BI dashboard development.

## Main Research Question

**What economic and demographic factors are connected to migration and brain drain from Albania?**

## Main Findings

### 1. Population decline is a central demographic issue

Albania’s population in 2000 was approximately **{fmt_number(pop_2000)}**. The latest available population value is **{fmt_number(latest_pop)}** in **{latest_pop_year}**.

This represents a change of approximately **{fmt_number(pop_change)}** people, or **{fmt_percent(pop_change_pct)}** compared with 2000.

This confirms that population decline is one of the most important background issues in the brain drain discussion.

### 2. Net migration shows migration pressure

The latest available World Bank net migration value is **{fmt_number(latest_net_migration)}** in **{latest_mig_year}**.

The strongest negative net migration year in the dataset is **{worst_mig_year}**, with a value of **{fmt_number(worst_mig_value)}**.

Negative net migration indicates that more people are leaving than entering during the measured period.

### 3. European destinations are important in Albanian migration patterns

Eurostat migration-flow data shows that Albanian citizens are recorded as arriving in several European reporting countries. The top recorded destination in the processed dashboard dataset is **{top_destination}**, with **{fmt_number(top_destination_value)}** recorded arrivals across the available period.

This data should be interpreted carefully because Eurostat immigration into European reporting countries is used as a proxy for Albanian international mobility, not as a direct count of everyone leaving Albania.

### 4. Residence permits help explain migration channels

Residence permit data helps separate migration reasons into categories such as employment, education, family, and other reasons.

The latest available residence permit year is **{latest_permits_year}**, with **{fmt_number(latest_permits)}** total first residence permits in the master dataset.

Residence permit reasons are important because they show that brain drain is connected not only to general migration, but also to labour-market opportunities, education pathways, and family networks.

### 5. Economy and employment are relevant but not the only explanation

The latest available unemployment rate is **{fmt_percent(latest_unemployment)}** in **{latest_unemp_year}**.

The latest available GDP growth rate is **{fmt_percent(latest_gdp_growth)}** in **{latest_gdp_year}**.

The latest available GDP per capita is **{fmt_number(latest_gdp_pc)} USD** in **{latest_gdppc_year}**.

These indicators help explain part of the migration pressure. However, migration cannot be explained by one economic factor only. Wage expectations, education quality, career opportunities, family networks, and destination-country policies also matter.

### 6. Demographic risk increases the importance of brain drain

The latest available working-age population is **{fmt_number(latest_working_age)}** in **{latest_working_year}**.

The latest available elderly share is **{fmt_percent(latest_elderly_share)}** in **{latest_elderly_year}**.

The latest available dependency ratio is **{fmt_decimal(latest_dependency)}** in **{latest_dependency_year}**.

This suggests that brain drain is not only a migration issue. It is also connected to the future size of the labour force, pension pressure, healthcare demand, and long-term economic development.

## Final Conclusion

The analysis suggests that Albania’s brain drain problem is connected to a combination of demographic decline, negative migration patterns, residence permit demand, employment conditions, income opportunities, and age-structure change.

The project does not claim that one factor alone causes migration. Instead, it shows that brain drain is a multi-dimensional issue involving economic, demographic, and social pressures.

## Main Limitations

- Eurostat migration data is a proxy for Albanian citizens’ mobility into European countries, not a complete direct measure of emigration from Albania.
- Residence permit totals and subcategories must be handled separately to avoid double counting.
- Some indicators have missing values for certain years.
- Correlation analysis does not prove causation.
- Forecasts are trend-based scenarios and should not be treated as exact predictions.
"""

# Research Report
research_report = f"""# Research Report

## Title

**Albania’s Brain Drain: A Data-Driven Analysis of Migration, Employment and Economic Development**

## Abstract

This project studies Albania’s brain drain using public datasets from the World Bank, Eurostat, population estimates, and official migration-related reports. The analysis focuses on population decline, net migration, European migration-flow indicators, first residence permits, unemployment, GDP growth, and demographic structure.

The project applies a low-level academic and portfolio-style data analytics workflow. Data was cleaned using Python and Pandas, structured into processed datasets, loaded into a PostgreSQL database, analyzed through exploratory and statistical methods, forecasted to 2035, and presented in a Power BI dashboard.

The findings suggest that Albania’s brain drain is connected to demographic decline, negative net migration, labour-market pressure, education and employment migration channels, and long-term changes in the working-age and elderly population.

## 1. Introduction

Brain drain refers to the movement of educated, skilled, or economically active people away from their home country. For Albania, this issue is important because migration can affect the labour force, education system, innovation capacity, tax base, public services, and long-term development.

This project studies Albania’s brain drain from a data analytics perspective. Instead of relying only on general opinion, it uses public datasets to examine population trends, migration indicators, residence permits, economic indicators, and demographic risk.

## 2. Problem Statement

Albania has experienced long-term demographic and migration challenges. Population decline, negative migration, and the movement of young and working-age people abroad can create pressure on the economy and society.

The main problem investigated in this project is whether economic and demographic indicators are connected to migration and brain drain from Albania.

## 3. Research Question

**What economic and demographic factors are connected to migration and brain drain from Albania?**

## 4. Research Objectives

The objectives of the project are:

1. Analyze Albania’s population trend since 2000.
2. Examine net migration and migration pressure.
3. Identify major European destination countries for Albanian citizens.
4. Analyze first residence permits by reason.
5. Study the connection between unemployment, GDP growth, and migration indicators.
6. Examine Albania’s youth, working-age, and elderly population structure.
7. Create simple forecasts to 2035.
8. Present the results in a Power BI dashboard.

## 5. Data Sources

The project uses the following data sources:

- World Bank Albania indicator dataset
- World Bank Population Estimates
- Eurostat migration-flow datasets
- Eurostat first residence permit datasets
- INSTAT and official population reports
- World Bank Albania migration survey/report context

The World Bank data was mainly used for population, GDP, unemployment, net migration, remittances, and FDI indicators. Eurostat data was used to study Albanian citizens arriving in European reporting countries and first residence permits issued to Albanian citizens.

## 6. Methodology

The project followed these steps:

1. Raw data collection
2. Dataset understanding
3. Data cleaning and transformation
4. Feature engineering
5. PostgreSQL database design
6. Exploratory data analysis
7. Statistical correlation analysis
8. Forecasting to 2035
9. Power BI dashboard development
10. Final reporting

Python and Pandas were used for cleaning and transformation. PostgreSQL was used to create a database with dimension and fact tables. Power BI was used to build the final dashboard.

## 7. Data Cleaning and Preparation

The World Bank data was transformed from wide format into long format and then reshaped into analysis-ready tables. Eurostat data was cleaned by filtering relevant rows and separating total categories from subcategories to reduce double counting risk.

A master analysis dataset was created by merging indicators by year. Additional features were created, including:

- Net migration rate per 1,000 population
- Residence permits per 100,000 population
- Employment permit share
- Education permit share
- Family permit share
- Migration pressure index
- Working-age and elderly population indicators

## 8. Database Design

The project includes a PostgreSQL database with dimension and fact tables.

Dimension tables include:

- date_dimension
- country_dimension
- indicator_dimension
- age_dimension
- migration_reason_dimension

Fact tables include:

- fact_population
- fact_economy
- fact_unemployment
- fact_net_migration
- fact_migration_flows
- fact_residence_permits

This structure supports analytical queries and shows how the cleaned data can be organized in a business-intelligence-ready format.

## 9. Exploratory Data Analysis

### Population

Albania’s population in 2000 was approximately **{fmt_number(pop_2000)}**. The latest available population value is **{fmt_number(latest_pop)}** in **{latest_pop_year}**.

The population change from 2000 to the latest available year is approximately **{fmt_number(pop_change)}**, or **{fmt_percent(pop_change_pct)}**.

This confirms that population decline is a major issue in the Albanian demographic context.

### Migration

The latest available net migration value is **{fmt_number(latest_net_migration)}** in **{latest_mig_year}**.

The strongest negative net migration year in the dataset is **{worst_mig_year}**, with a value of **{fmt_number(worst_mig_value)}**.

Negative net migration is a sign of migration pressure and population loss.

### Destination Countries

The processed Eurostat migration data identifies **{top_destination}** as the top recorded destination country in the available dataset, with **{fmt_number(top_destination_value)}** recorded immigration observations.

This should be interpreted as Albanian citizens recorded as arriving in European reporting countries, not as a complete direct count of emigration from Albania.

### Residence Permits

Residence permit data helps explain migration channels. It separates migration into reasons such as employment, education, family, and other categories.

Latest available total first residence permits in the master dataset: **{fmt_number(latest_permits)}** in **{latest_permits_year}**.

Latest residence permit reason breakdown:

{safe_markdown(latest_reason_table.head(10))}

## 10. Economy and Employment

The latest available unemployment rate is **{fmt_percent(latest_unemployment)}** in **{latest_unemp_year}**.

The latest available GDP growth rate is **{fmt_percent(latest_gdp_growth)}** in **{latest_gdp_year}**.

The latest available GDP per capita is **{fmt_number(latest_gdp_pc)} USD** in **{latest_gdppc_year}**.

These indicators are relevant because weak employment opportunities, income gaps, and limited career growth can encourage migration. However, economic indicators alone do not fully explain brain drain.

## 11. Statistical Analysis

The project tested relationships between migration indicators and economic indicators using Pearson and Spearman correlation.

Correlation results:

{safe_markdown(correlation_brief)}

The statistical results should be interpreted carefully. Correlation does not prove causation. A statistically significant relationship only shows that two variables move together in the available data, not that one directly causes the other.

## 12. Demographic Structure

The latest available working-age population is **{fmt_number(latest_working_age)}** in **{latest_working_year}**.

The latest available elderly share is **{fmt_percent(latest_elderly_share)}** in **{latest_elderly_year}**.

The latest available dependency ratio is **{fmt_decimal(latest_dependency)}** in **{latest_dependency_year}**.

This part of the analysis is important because brain drain can reduce the labour force while the elderly population grows. This can affect pensions, healthcare, tax revenue, public services, and long-term productivity.

## 13. Forecasting

The project created simple forecasts to 2035 using trend-based models such as Linear Regression, Moving Average baseline, and ARIMA.

2035 forecast summary:

{safe_markdown(forecast_brief)}

Forecasts should be interpreted as scenarios, not exact predictions. Migration and population trends can change due to policy decisions, economic shocks, education opportunities, labour-market conditions, and destination-country rules.

## 14. Power BI Dashboard Summary

The Power BI dashboard contains five pages:

1. Executive Summary
2. Migration Analysis
3. Economy and Employment
4. Demographic Risk
5. Forecasting

The dashboard visualizes the main population, migration, economy, employment, residence permit, demographic, and forecasting results.

## 15. Key Findings

The main findings are:

1. Albania has experienced population decline since 2000.
2. Net migration shows migration pressure in the available data.
3. European destination countries are important in Albanian migration patterns.
4. Residence permit data shows that migration is connected to employment, education, and family reasons.
5. Economic indicators such as unemployment and GDP growth help explain part of the migration context.
6. The working-age and elderly population structure creates long-term demographic risk.
7. Forecasts suggest that demographic and migration pressure may continue if historical trends continue.

## 16. Policy Recommendations

Based on the analysis, possible policy recommendations include:

1. Improve youth employment opportunities.
2. Increase wages and career development pathways.
3. Strengthen higher education and professional training.
4. Support return migration and diaspora engagement.
5. Create incentives for skilled workers to remain in Albania.
6. Improve regional development outside Tirana.
7. Use migration and labour-market data for evidence-based policymaking.

## 17. Limitations

This project has several limitations:

- Eurostat immigration data records Albanian citizens arriving in European reporting countries. It is used as a proxy for mobility, not a direct measure of all emigration from Albania.
- Residence permit TOTAL categories and subcategories must be handled separately to avoid double counting.
- Some indicators have missing values.
- Correlation analysis does not prove causation.
- Forecasts are trend-based and should not be interpreted as exact future outcomes.
- The project is a low-level academic and portfolio-style analysis, not a published scientific study.

## 18. Conclusion

This project shows that Albania’s brain drain is connected to a combination of demographic, economic, and migration-related factors. Population decline, negative net migration, residence permit demand, unemployment, GDP trends, and changing age structure all contribute to the broader issue.

The project demonstrates the full data analytics lifecycle, from raw public data to cleaning, database design, analysis, forecasting, dashboard development, and final reporting.

## 19. References

- World Bank Open Data
- World Bank Population Estimates
- Eurostat migration flow datasets
- Eurostat residence permit datasets
- INSTAT population and census reports
- World Bank Albania migration survey/report

## 20. Appendix

Project outputs include:

- Cleaned CSV files
- PostgreSQL schema and SQL queries
- Jupyter notebooks
- Exploratory charts
- Statistical analysis results
- Forecasting results
- Power BI dashboard
- README and LinkedIn materials
"""

# Save files
executive_path = REPORTS_DIR / "executive_summary.md"
research_path = REPORTS_DIR / "research_report.md"

with open(executive_path, "w", encoding="utf-8") as f:
    f.write(executive_summary)

with open(research_path, "w", encoding="utf-8") as f:
    f.write(research_report)

print("Saved:", executive_path)
print("Saved:", research_path)

Saved: c:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\reports\executive_summary.md
Saved: c:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\reports\research_report.md
